# Assignment 1: Market Clearing (System Perspective)

### Import external libraries

In [1]:
import gurobipy as gp
from gurobipy import GRB
import pandas as pd
import numpy as np 
import matplotlib.pyplot as plt
import network
import networkx as nx
from network import initialize_network, system_demand, LINES

import pypsa
import math


E0000 00:00:1773741081.271408 1487485 instrument.cc:563] Metric with name 'grpc.resource_quota.calls_dropped' registered more than once. Ignoring later registration.
E0000 00:00:1773741081.271538 1487485 instrument.cc:563] Metric with name 'grpc.resource_quota.calls_rejected' registered more than once. Ignoring later registration.
E0000 00:00:1773741081.271542 1487485 instrument.cc:563] Metric with name 'grpc.resource_quota.connections_dropped' registered more than once. Ignoring later registration.
E0000 00:00:1773741081.271552 1487485 instrument.cc:563] Metric with name 'grpc.resource_quota.instantaneous_memory_pressure' registered more than once. Ignoring later registration.
E0000 00:00:1773741081.271554 1487485 instrument.cc:563] Metric with name 'grpc.resource_quota.memory_pressure_control_value' registered more than once. Ignoring later registration.
/Users/carloshunziker/opt/miniconda3/envs/infra/lib/python3.10/site-packages/google/api_core/_python_version_support.py:275: Future

## Step 0: Build a Relevant Case Study

Please select an electric power network from the following options:
1. IEEE 24-bus reliability test system: link.
2. IEEE reliability test system (2019 update): link.
3. IEEE power systems test cases (various cases with 14, 30, 57, 118, and 300 buses): link.
4. Case studies available in the open-source Julia platform PowerModels.jl: link.

You are also free to choose another case study. If some data is missing, please select reason-
able arbitrary values. For technical details on conventional generators and transmission lines, this link may be helpful (it corresponds to the IEEE 24-bus case study, but similar data can be used for other cases).

### IEEE 24-bus reliability test system

In [2]:
consumers, generators = initialize_network()
print(generators)
print(consumers)
print("Network initialized successfully.")

[Generator(unit=1, node=1), Generator(unit=2, node=2), Generator(unit=3, node=7), Generator(unit=4, node=13), Generator(unit=5, node=15), Generator(unit=6, node=15), Generator(unit=7, node=16), Generator(unit=8, node=18), Generator(unit=9, node=21), Generator(unit=10, node=22), Generator(unit=11, node=23), Generator(unit=12, node=23), Generator(unit=13, node=3), Generator(unit=14, node=5), Generator(unit=15, node=7), Generator(unit=16, node=16), Generator(unit=17, node=21), Generator(unit=18, node=23)]
[Consumer(load=1, node=1, share=0.038), Consumer(load=2, node=2, share=0.034), Consumer(load=3, node=3, share=0.063), Consumer(load=4, node=4, share=0.026), Consumer(load=5, node=5, share=0.025), Consumer(load=6, node=6, share=0.048), Consumer(load=7, node=7, share=0.044), Consumer(load=8, node=8, share=0.06), Consumer(load=9, node=9, share=0.061), Consumer(load=10, node=10, share=0.068), Consumer(load=11, node=13, share=0.093), Consumer(load=12, node=14, share=0.068), Consumer(load=13, 

### Additional Assumptions


*   Assume that the price bids of all producers are non-negative and equal to their marginal production cost. In particular, the production cost of renewable units is assumed to be zero. Additionally, these units offer their forecasted capacity, meaning their offer quantities vary over time.

*   For the bid price of price-elastic demands, use comparatively high values (relative to the generation cost of conventional units) to ensure that most demands are supplied. For inspiration, check the real bid price data in Nord Pool [link].
*   A potential source for wind power forecast data is available at this link (you may nor- malize the data to fit your case study). Another potential source for the renewable power generation data is renewables.ninja.
*   For transmission lines, you may assume a uniform reactance for all lines (e.g., 0.002 p.u., leading to a susceptance of 500 p.u.).

### The market-clearing price under a uniform pricing scheme.

In [3]:
# define the demand time at 16:00
total_consumption_16 = system_demand[16]
print(f"Total consumption at 16:00 is {total_consumption_16} MW")
#check if all generators are there
id_Gnerators = []
for i in generators:
    id_Gnerators.append(i.unit_id)
print(id_Gnerators)

Total consumption at 16:00 is 2464.965 MW
[1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 12, 13, 14, 15, 16, 17, 18]


Define the supply


In [4]:
#Variable generators costs 
generator_cost = []
for i in generators:
    generator_cost.append(i.cost_energy)
print(generator_cost)

#variable demand bids prize
demand_bids = []
#consumption for each consumer
consumption = []
for i in consumers:
    demand_bids.append(i.price)
    consumption.append(i.share * total_consumption_16)
print(demand_bids)
print(consumption)

if sum(consumption) == total_consumption_16:
    print("Aggregation correct")

[13.32, 13.32, 20.7, 20.93, 26.11, 10.52, 10.52, 6.02, 5.47, 0, 10.52, 10.89, 0, 0, 0, 0, 0, 0]
[165.0, 155.0, 175.0, 150.0, 145.0, 170.0, 160.0, 180.0, 185.0, 190.0, 210.0, 195.0, 220.0, 158.0, 215.0, 178.0, 162.0]
[93.66867, 83.80881000000001, 155.292795, 64.08909, 61.62412500000001, 118.31832000000001, 108.45846, 147.8979, 150.362865, 167.61762000000002, 229.241745, 167.61762000000002, 273.61111500000004, 86.27377500000001, 288.400905, 157.75776000000002, 110.92342500000001]
Aggregation correct


Define Generator capacity:

In [5]:
#Generators capacity
generator_capacity = []
for i in generators:
    generator_capacity.append(i.p_max)
print(generator_capacity)

[152, 152, 350, 591, 60, 155, 155, 400, 400, 300, 310, 350, 200, 200, 200, 200, 200, 200]


## Step 5: Balancing Market

In Lecture 5, you learn about the balancing market. Building upon the model in Step1 (no storage, no transmission network, 1 hour), assume there is an unexcepted failure (outage) in one of the conventional generaors. Additionally, the actual production of some wind farms is lower than their day-ahead forcast (e.g., 15%), while that of the remaining wind farms is higher than their day-ahead forcast (e.g., 10%). A subset of conventional generators are potential balancing service providers. Demands are not felxible. \\

If possible (depending on their day-ahead schedule), each felxible conventional generator offers upward regulation service at a price equal to the day-ahead price plus 10% of its production cost. Similarly, it offers downward regulation service at a price equal to te day-ahead price minus 15% of its production cost. The load curtailment cost is 500/MWh.

How does the reserve market change prices in the day-ahead market?

Please clear the balancing market for the given hour and derive the balancing price. Then,
calculate the total profit (in both day-ahead and balancing markets) of all conventional gener-
ators and wind farms in the given hour, assuming the imbalance settlement follows:
1. One-price scheme,
2. Two-price scheme. 

By comparing these profits, please analyze the implications of these two schemes for those
who provide balancing services and for those who cause the imbalance.

In [3]:
"""
This script simulates an electricity market with three main steps:

1. Day-Ahead market:
   Generators submit planned production.

2. Real-time deviations:
   - One generator outage
   - Wind forecast errors (under/overproduction)

3. Balancing market:
   Flexible generators compensate deviations via up/down regulation.
   Profits are then calculated under:
   - one-price imbalance settlement
   - two-price imbalance settlement
"""

# ------------------------------------------------------------------
# 1) INPUT DATA
# ------------------------------------------------------------------

# Day-Ahead market price ($/MWh)
lambda_DA = 10.52

# Scheduled generation in the Day-Ahead market (MW)
p_DA = {
    1: 0.0,   2: 0.0,   3: 0.0,   4: 0.0,   5: 0.0,
    6: 0.0,   7: 0.0,   8: 400.0, 9: 400.0, 10: 300.0,
    11: 164.965, 12: 0.0,
    13: 200.0, 14: 200.0, 15: 200.0, 16: 200.0, 17: 200.0, 18: 200.0
}

# Maximum generation capacity (MW)
Pmax = {
    1: 152.0, 2: 152.0, 3: 350.0, 4: 591.0, 5: 60.0, 6: 155.0,
    7: 155.0, 8: 400.0, 9: 400.0, 10: 300.0, 11: 310.0, 12: 350.0,
    13: 200.0, 14: 200.0, 15: 200.0, 16: 200.0, 17: 200.0, 18: 200.0
}

# Marginal production costs ($/MWh)
Cg = {
    1: 13.32, 2: 13.32, 3: 20.70, 4: 20.93, 5: 26.11, 6: 10.52,
    7: 10.52, 8: 6.02, 9: 5.47, 10: 0.00, 11: 10.52, 12: 10.89,
    13: 0.00, 14: 0.00, 15: 0.00, 16: 0.00, 17: 0.00, 18: 0.00
}

# Generator outage (fails in real time)
outage_gen = 11

# Wind generators with lower-than-forecast production (−15%)
low_wind = [13, 14, 15]

# Wind generators with higher-than-forecast production (+10%)
high_wind = [16, 17, 18]

# Generators able to provide balancing services
flex = [1, 2, 3, 4, 5, 6, 7, 11, 12]


# ------------------------------------------------------------------
# 2) REAL-TIME PRE-BALANCING GENERATION
# ------------------------------------------------------------------

# Start from Day-Ahead schedule
p_RT_pre = p_DA.copy()

# Apply generator outage
p_RT_pre[outage_gen] = 0.0

# Apply wind forecast errors
for g in low_wind:
    p_RT_pre[g] = 0.85 * p_DA[g]

for g in high_wind:
    p_RT_pre[g] = 1.10 * p_DA[g]

# Total system imbalance:
# positive  -> generation deficit (need upward regulation)
# negative  -> generation surplus (need downward regulation)
balancing_need = sum(p_DA[g] - p_RT_pre[g] for g in p_DA)

print(f"Balancing need = {balancing_need:.3f} MW")


# ------------------------------------------------------------------
# 3) BALANCING MARKET OFFERS
# ------------------------------------------------------------------

# Upward regulation offers (increase generation)
up_offer = {g: lambda_DA + 0.1 * Cg[g] for g in flex}

# Downward regulation offers (reduce generation)
down_offer = {g: lambda_DA - 0.15 * Cg[g] for g in flex}

# Available upward capacity (headroom)
up_cap = {g: Pmax[g] - p_DA[g] for g in flex}

# Available downward capacity (current output)
down_cap = {g: p_DA[g] for g in flex}

# Remove failed generator from balancing participation
available_flex = [g for g in flex if g != outage_gen]


# ------------------------------------------------------------------
# 4) CLEAR BALANCING MARKET
# ------------------------------------------------------------------

# Activated balancing volumes
r_up = {g: 0.0 for g in p_DA}
r_down = {g: 0.0 for g in p_DA}

# Unserved demand (if system cannot balance)
load_curtailment = 0.0

# Case A: upward regulation needed
if balancing_need > 0:
    # Merit order: cheapest offers first
    merit_order = sorted(available_flex, key=lambda g: (up_offer[g], g))
    remaining = balancing_need

    for g in merit_order:
        activated = min(up_cap[g], remaining)
        r_up[g] = activated
        remaining -= activated

        if remaining <= 1e-9:
            break

    # If not enough capacity → load shedding
    if remaining > 1e-9:
        load_curtailment = remaining
        lambda_B = 500.0  # penalty price
    else:
        # Marginal unit sets the price
        marginal_gen = max(
            [g for g in merit_order if r_up[g] > 0],
            key=lambda g: up_offer[g]
        )
        lambda_B = up_offer[marginal_gen]

# Case B: downward regulation needed
elif balancing_need < 0:
    merit_order = sorted(
        available_flex,
        key=lambda g: (down_offer[g], g),
        reverse=True
    )
    remaining = -balancing_need

    for g in merit_order:
        activated = min(down_cap[g], remaining)
        r_down[g] = activated
        remaining -= activated

        if remaining <= 1e-9:
            break

    marginal_gen = min(
        [g for g in merit_order if r_down[g] > 0],
        key=lambda g: down_offer[g]
    )
    lambda_B = down_offer[marginal_gen]

# Case C: no imbalance
else:
    lambda_B = lambda_DA

print(f"Balancing price = {lambda_B:.3f} $/MWh")


# ------------------------------------------------------------------
# 5) REAL-TIME GENERATION AFTER BALANCING
# ------------------------------------------------------------------

p_actual = p_RT_pre.copy()

for g in p_actual:
    p_actual[g] += r_up[g]
    p_actual[g] -= r_down[g]


# ------------------------------------------------------------------
# 6) PROFIT CALCULATION
# ------------------------------------------------------------------

profit_one = {}  # one-price system
profit_two = {}  # two-price system

for g in p_DA:
    # Day-Ahead revenue
    revenue_DA = lambda_DA * p_DA[g]

    # Balancing market revenue
    revenue_BSP = lambda_B * r_up[g] - lambda_B * r_down[g]

    # Net imbalance (not covered by balancing activation)
    deviation = p_actual[g] - p_DA[g] - r_up[g] + r_down[g]

    # Production cost
    production_cost = Cg[g] * p_actual[g]

    # -------------------------
    # One-price system
    # -------------------------
    imbalance_payment_one = lambda_B * deviation

    profit_one[g] = (
        revenue_DA + revenue_BSP + imbalance_payment_one - production_cost
    )

    # -------------------------
    # Two-price system
    # -------------------------
    if balancing_need > 0:
        settlement_price = lambda_B if deviation < 0 else lambda_DA
    elif balancing_need < 0:
        settlement_price = lambda_B if deviation > 0 else lambda_DA
    else:
        settlement_price = lambda_DA

    imbalance_payment_two = settlement_price * deviation

    profit_two[g] = (
        revenue_DA + revenue_BSP + imbalance_payment_two - production_cost
    )


# ------------------------------------------------------------------
# 7) OUTPUT RESULTS
# ------------------------------------------------------------------

print("\nProfits under one-price and two-price:")
for g in range(1, 19):
    print(
        f"G{g:02d}: "
        f"one-price = {profit_one[g]:10.3f} $   "
        f"two-price = {profit_two[g]:10.3f} $"
    )

Balancing need = 194.965 MW
Balancing price = 11.572 $/MWh

Profits under one-price and two-price:
G01: one-price =      0.000 $   two-price =      0.000 $
G02: one-price =      0.000 $   two-price =      0.000 $
G03: one-price =      0.000 $   two-price =      0.000 $
G04: one-price =      0.000 $   two-price =      0.000 $
G05: one-price =      0.000 $   two-price =      0.000 $
G06: one-price =    163.060 $   two-price =    163.060 $
G07: one-price =     42.043 $   two-price =     42.043 $
G08: one-price =   1800.000 $   two-price =   1800.000 $
G09: one-price =   2020.000 $   two-price =   2020.000 $
G10: one-price =   3156.000 $   two-price =   3156.000 $
G11: one-price =   -173.543 $   two-price =   -173.543 $
G12: one-price =      0.000 $   two-price =      0.000 $
G13: one-price =   1756.840 $   two-price =   1756.840 $
G14: one-price =   1756.840 $   two-price =   1756.840 $
G15: one-price =   1756.840 $   two-price =   1756.840 $
G16: one-price =   2335.440 $   two-price =   